In [ ]:

!pip install medmnist

import torch
import torch.nn as nn
import torch.optim as optim
import medmnist
from medmnist import PneumoniaMNIST
from torch.utils.data import DataLoader
import torchvision.transforms as transforms


data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

train_dataset = PneumoniaMNIST(split='train', transform=data_transform, download=True)
test_dataset = PneumoniaMNIST(split='test', transform=data_transform, download=True)


train_loader = DataLoader(dataset=train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=128, shuffle=False)

print(f"Dataset Loaded! Training Images: {len(train_dataset)}, Test Images: {len(test_dataset)}")

class MedicalCNN(nn.Module):
    def __init__(self):
        super(MedicalCNN, self).__init__()

        # Conv2d(input_channels=1 (grayscale), output_channels=16, kernel_size=3)
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # Shrinks image by half
        )

        self.layer2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # Shrinks again
        )

        # The image started as 28x28.
        # After Pool 1 -> 14x14. After Pool 2 -> 7x7.
        # So input size is 32 channels * 7 * 7
        self.fc = nn.Linear(32 * 7 * 7, 1) # Binary Classification (0 or 1)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MedicalCNN().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"\nStarting Training on {device}...")

for epoch in range(3):
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device).float()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch+1}/3], Loss: {loss.item():.4f}")

print("Medical CV Task Complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 3.6 MB/s eta 0:00:00


100%|██████████| 4.17M/4.17M [00:00<00:00, 4.35MB/s]


Dataset Loaded! Training Images: 4708, Test Images: 624

Starting Training on cpu...
Epoch [1/3], Loss: 0.1577
Epoch [2/3], Loss: 0.1200
Epoch [3/3], Loss: 0.1490
Medical CV Task Complete!
